In [1]:
import os
import json
import requests
import gradio as gr
import ollama

OLLAMA_MODEL = "qwen3:8b"
WEATHERAPI_KEY = "d1d1"
WEATHERAPI_URL = "http://api.weatherapi.com/v1/current.json"

In [19]:
def get_weather(city: str) -> dict:
        r = requests.get(
            WEATHERAPI_URL,
            params={"key": WEATHERAPI_KEY, "q": city ,"aqi": "no"},
        )
        data = r.json()
        loc, cur = data["location"], data["current"]
        return {
            "city": loc["name"],
            "country": loc["country"],
            "temp_c": cur["temp_c"],
            "feelslike_c": cur["feelslike_c"],
            "condition": cur["condition"]["text"],
            "humidity": cur["humidity"],
            "wind_kph": cur["wind_kph"],
        }



In [16]:
EXTRACT_PROMPT = """extract structured info from a user message.
Decide if the user is asking about current weather for a specific place.
Respond ONLY with valid JSON (no extra text) in this exact format:
{{"is_weather_question": true or false, "city": "city name or empty string"}}
This is the user message:"""

def extract_weather_intent(message):
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "user", "content": EXTRACT_PROMPT + message}
            ],
            format="json",
            options={"temperature": 0},
        )
        return json.loads(response["message"]["content"])

In [17]:
def chat_fn(message, history):
    intent = extract_weather_intent(message) #dic

    if intent.get("is_weather_question") and intent.get("city"):
        weather = get_weather(intent["city"])
        context = (
                f"Current weather in {weather['city']}, {weather['country']}: "
                f"{weather['condition']}, {weather['temp_c']}°C "
                f"(feels like {weather['feelslike_c']}°C)"
        )
        system = (
            "You are a friendly weather assistant. Use the following live weather "
            f"data to answer the user's question naturally and concisely:\n{context}"
        )
    else:
        system = (
            "You are a friendly general assistant. If the user asks about weather without naming a city, ask which city they mean. Otherwise, chat normally."
        )

    messages = [{"role": "system", "content": system}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    response = ollama.chat(model=OLLAMA_MODEL, messages=messages)
    return response["message"]["content"]

In [20]:
chat_fn("What is the weather in cairo?",[])

'The weather in Cairo, Egypt is currently clear with a temperature of 27.3°C. It feels slightly warmer at 27.8°C, so you might want to stay hydrated and enjoy the sunshine! 🌞'

In [21]:
import gradio as gr

demo = gr.ChatInterface(fn=chat_fn, examples=["What's the weather in Cairo?", "is it raining in Tokyo?"], title="Weather Bot")
demo.launch()


c:\Users\GIGABYTE\Downloads\summariz ollama\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
